# Silver · clientes

**Origem:** `vcredit_bronze.clientes` → **Destino:** `vcredit_silver.clientes`

**Grão:** uma linha por cliente. **Depende de:** bronze

Cadastro limpo dos clientes, com região, idade, data de cadastro e dia de vencimento da fatura.

| Regra | Tratamento |
|---|---|
| `idade` | Convertida para número; clientes com idade ausente ou fora de 18 a 110 são removidos |
| `id_cliente` | Espaços sobrando removidos; mantido como texto (preserva zeros à esquerda); ID repetido mantém um registro |
| `nome` | Espaços sobrando removidos; iniciais maiúsculas |
| `email` | Espaços internos removidos ("@hotmail . com"), minúsculo; formato inválido vira nulo |
| `regiao` | Minúsculo, sem espaços sobrando |
| `faixa_etaria` | Criada a partir da idade (18-24, 25-34, 35-44, 45-54, 55-64, 65+) |
| `data_cadastro` | Convertida para data (aceita DD/MM/YYYY e YYYY-MM-DD) |
| `dia_vencimento_fatura` | Convertido para número; fora de 1 a 31 vira nulo |

**Decisões:** a faixa etária permite analisar perfis na gold sem expor a idade exata do cliente. Os clientes removidos pela regra de idade deixam seus chamados sem cliente correspondente; na gold, esses chamados ficam como "cliente não identificado".

Pipeline idempotente (lê da bronze e sobrescreve).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "rocket_lab"

bronze_schema_name = "vcredit_bronze"
bronze_schema = f"{catalog}.{bronze_schema_name}"

silver_schema_name = "vcredit_silver"
silver_schema = f"{catalog}.{silver_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

In [0]:
# Importação da tabela clientes da camada bronze
bronze_clientes = spark.table(f"{bronze_schema}.clientes")

silver_clientes = (
    bronze_clientes
    .withColumn("idade", F.expr("try_cast(trim(idade) as int)"))
    .where((F.col("idade").isNotNull()) & (F.col("idade").between(18, 110)))
    .withColumn("id_cliente", F.trim("id_cliente"))
    .dropDuplicates(["id_cliente"])
    .withColumn("nome", F.initcap(F.regexp_replace(F.trim("nome"), r"\s+", " ")))

    # remove espaços internos ("@hotmail . com") e valida o formato
    .withColumn("email", F.lower(F.regexp_replace("email", r"\s+", "")))
    .withColumn("email", F.when(F.col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"), F.col("email")).otherwise(None))
    .withColumn("regiao", F.lower(F.trim(F.col("regiao"))))
    
    # cria a coluna para a faixa etária 
    .withColumn("faixa_etaria",
        F.when(F.col("idade") < 25, "18-24").when(F.col("idade") < 35, "25-34")
         .when(F.col("idade") < 45, "35-44").when(F.col("idade") < 55, "45-54")
         .when(F.col("idade") < 65, "55-64").otherwise("65+"))
    .withColumn("data_cadastro", F.coalesce(
        F.expr("try_to_date(data_cadastro, 'dd/MM/yyyy')"),
        F.expr("try_to_date(data_cadastro, 'yyyy-MM-dd')")))
    .withColumn("dia_vencimento_fatura", F.expr("try_cast(dia_vencimento_fatura as int)"))
    .withColumn("dia_vencimento_fatura", F.when((F.col("dia_vencimento_fatura") <= 0) | (F.col("dia_vencimento_fatura") > 31), None).otherwise(F.col("dia_vencimento_fatura")))
)

 # Idempotente
(
    silver_clientes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.clientes")
)

display(silver_clientes.orderBy("id_cliente").limit(10))